# Visualizing and Tracing a Graph

Building a graph is only half the job. This notebook covers three ways to see what a compiled graph is actually doing: drawing its structure as a diagram, inspecting a single run step by step, and turning on LangSmith tracing to see every run in a web UI. It reuses the same tool-calling chatbot from `15_Tools`.

In [ ]:
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

## Part 1: Rebuilding the Tool-Calling Chatbot

This is the same graph as `15_Tools/Tooling.ipynb`: a chat node bound to a calculator tool, a `ToolNode` to run it, and an edge from `tools` back to `chat_node` so the model gets to read the tool's result.

In [ ]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """Performs basic arithmetic operations on two numbers."""
    if operation == "add":
        result = first_num + second_num
    elif operation == "subtract":
        result = first_num - second_num
    elif operation == "multiply":
        result = first_num * second_num
    elif operation == "divide":
        if second_num == 0:
            return {"error": "Cannot divide by zero."}
        result = first_num / second_num
    else:
        return {"error": "Invalid operation. Supported operations are add, subtract, multiply, divide."}
    return {"result": result}


tools = [calculator]
llm_with_tools = llm.bind_tools(tools)


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

app = graph.compile()

## Part 2: Drawing the Graph as Mermaid Text

`app.get_graph()` returns a graph representation you can inspect without running anything. `.draw_mermaid()` turns it into Mermaid diagram syntax as plain text, which renders as a diagram on GitHub, in most Markdown viewers, and in the Mermaid Live Editor (no network call needed to produce the text).

In [ ]:
print(app.get_graph().draw_mermaid())

## Part 3: Rendering a PNG (Needs Network)

`.draw_mermaid_png()` renders the same diagram to an image, but by default it calls a hosted Mermaid rendering service over the network to do it. If this cell fails or hangs, it usually means there is no network access from this machine; the Mermaid text from Part 2 works everywhere and needs no network.

In [ ]:
try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"draw_mermaid_png needs network access to a Mermaid rendering service: {e}")

## Part 4: Inspecting a Run With `stream_mode`

`invoke` only returns the final state. `stream` with `stream_mode="debug"` yields one event per internal step of a run: which task started, which task finished, and what it wrote. This is the same information LangSmith traces show, without needing an account. `stream_mode="tasks"` is narrower: it only yields task start/result events, without the extra checkpoint bookkeeping `"debug"` includes.

In [ ]:
for event in app.stream(
    {"messages": [HumanMessage(content="In one short sentence, what is LangGraph?")]},
    stream_mode="debug",
):
    print(event["type"], "-", event["payload"].get("name", ""))

In [ ]:
for event in app.stream(
    {"messages": [HumanMessage(content="In one short sentence, what is LangGraph?")]},
    stream_mode="tasks",
):
    print(event)

## Part 5: Turning On LangSmith Tracing (Optional)

Tracing is off unless you set environment variables for it. It is not required to run anything in this notebook or this repo.

```text
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=<your key>
LANGSMITH_PROJECT=<a project name, optional>
```

Set these in `.env` (or your shell environment) before calling `load_dotenv()`, and every `invoke`/`stream` call after that is traced automatically; nothing in your graph code has to change. Since this notebook does not set them, the calls above ran untraced.

## Summary

- `app.get_graph().draw_mermaid()` prints a diagram of the compiled graph as text; no network needed.
- `app.get_graph().draw_mermaid_png()` renders the same diagram as an image, but needs network access to a Mermaid rendering service.
- `stream_mode="debug"` shows every internal step of a run; `stream_mode="tasks"` narrows that to just task start/result events.
- LangSmith tracing turns on with three environment variables and needs no code changes; it is optional everywhere in this repo.
- The `app/` folder next to this notebook packages the same kind of graph for deployment; see the guide for `langgraph dev`, `langgraph build`/`up`, and calling a running server with `langgraph-sdk`.